# Seleccion del Mejor Modelo con PyCaret

Este notebook implementa un flujo completo de selección y comparación automática de modelos de Machine Learning utilizando PyCaret sobre el dataset `data/train.csv` usando la variable `SalePrice` como target. Se sigue el proceso estándar: carga de datos, setup del experimento, comparación de modelos (al menos 20), tuning de hiperparámetros, finalización y guardado del pipeline ganador. Todas las decisiones y resultados están documentadas paso a paso.

---

## 1. Imports y carga de datos

In [ ]:
import pandas as pd
import pycaret
import joblib

data = pd.read_csv('data/train.csv')
data.shape, data.columns

---

## 2. Determinación del tipo de problema

La variable target `SalePrice` es **numérica continua** (como se evidenció en el EDA), por lo tanto, este es un problema de **regresión**.

---

## 3. Import del experimento correcto

Usamos el motor de experimentos de regresión, ya que `SalePrice` es continua.

In [ ]:
from pycaret.regression import RegressionExperiment

Experimento = RegressionExperiment()

---

## 4. Setup del experimento

In [ ]:
Experimento.setup(
    data=data, 
    target='SalePrice', 
    session_id=123,
    verbose=True # Nos ayuda a ver el reporte completo
)

En este setup:
- PyCaret automáticamente imputará missing values, codificará variables categóricas y realizará validación cruzada KFold para asegurar estimaciones robustas de la métrica del modelo.

---

## 5. Comparación automática de modelos

Comparamos una amplia variedad de algoritmos de regresión con validación cruzada:

In [ ]:
MejorModelo = Experimento.compare_models()
print(MejorModelo)

Esto nos da el mejor modelo según la métrica de error cuadrático medio (MSE) por defecto en regresión.

---

## 6. Comparación específica (Top 3 modelos)

Seleccionamos explícitamente los 3 mejores modelos ordenados por la métrica principal (**MSE**):

In [ ]:
Mejores_TOP3 = Experimento.compare_models(sort='MSE', n_select=3)
print(Mejores_TOP3)

Esto nos permite analizar no solo el mejor sino varios candidatos cercanos.

---

## 7. Tuning de hiperparámetros del mejor modelo

Ahora optimizamos el mejor modelo encontrado utilizando búsqueda aleatoria (10 iteraciones):

In [ ]:
MejorModelo_Tuneado = Experimento.tune_model(estimator=MejorModelo, n_iter=10)
Experimento.plot_model(MejorModelo_Tuneado, plot='error')  # Visualiza errores
Experimento.plot_model(MejorModelo_Tuneado, plot='residuals')  # Visualiza residuos

Tras el tuning, PyCaret muestra las nuevas métricas. Si la mejora no es significativa, el modelo base puede ser preferible. Se recomienda revisar las tablas comparativas y los gráficos generados.

---

## 8. Finalización del modelo para producción

Preparar el pipeline para producción (transformaciones + modelo):

In [ ]:
Modelo_Final = Experimento.finalize_model(estimator=MejorModelo_Tuneado)

Esto congela todo el pipeline de datos y entrenamientos tal como sería usado en nuevos datos.

---

## 9. Guardar el modelo de dos formas

Guardamos tanto el pipeline completo (transformación + modelo) en formato de PyCaret como el modelo tuneado en formato `joblib` puro:

In [ ]:
# Formato pipeline completo PyCaret
Experimento.save_model(model=Modelo_Final, model_name='proyecto_pipeline_machine_learning_muestra/Modelo_Final_PyCaret')

# Formato puro (solo el modelo entrenado)
joblib.dump(MejorModelo_Tuneado, 'proyecto_pipeline_machine_learning_muestra/modelo_entrenado_solo.pkl')

Elegir entre ambos depende de si luego querrás reutilizar todo el pipeline de transformación o solo el algoritmo ya ajustado.

---

## 10. Cargar y verificar el modelo guardado

Como verificación, cargamos el pipeline guardado y comprobamos que funciona.

In [ ]:
Modelo_Cargado = Experimento.load_model('proyecto_pipeline_machine_learning_muestra/Modelo_Final_PyCaret')
print(type(Modelo_Cargado))

---

# Conclusión

Este flujo automatizado permite seleccionar, afinar y exportar de forma reproducible el mejor modelo de ML posible para predecir `SalePrice` con PyCaret. Todos los pasos están documentados y listos para modificar según necesidades avanzadas.